# Generación de datasets con inconsistencias

**Objetivo:** A partir de las muestras de 450.000 registros generadas en el Notebook 01, crear dos datasets ficticios:
- `dataset_2020_sucio.csv` (derivado de los años 2021 y 2022)
- `dataset_2025_sucio.csv` (derivado de los años 2023 y 2024)

Se introducen inconsistencias realistas de baja complejidad para practica de limpieza en KNIME.

**Inconsistencias aplicadas (5-10% de registros):**
- Valores nulos en campos de texto
- Espacios al inicio o final del texto
- Diferencias entre mayúsculas y minúsculas
- Valores "No registra", "N/A", "-", ""
- Duplicados parciales (filas repetidas in-place)

**Restricciones:**
- No modificar la cantidad de registros (450.000 fijos)
- No eliminar ni agregar columnas
- No alterar la estructura del dataset

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import os
import gc
from glob import glob

## 2. Configuración

In [2]:
CARPETA = os.getcwd()
RANDOM_STATE = 42
N_REGISTROS = 450_000
N_POR_ANIO = N_REGISTROS // 2  # 225.000 registros por año de origen

# Porcentaje de registros con inconsistencias (aprox 5-10%)
PCT_NULOS = 0.025          # 2.5% valores nulos
PCT_ESPACIOS = 0.02        # 2% espacios extra
PCT_MAYUSCULAS = 0.02      # 2% mayúsculas/minúsculas mixtas
PCT_VALORES_REEMPLAZO = 0.02  # 2% "No registra", "N/A", "-", ""
PCT_DUPLICADOS = 0.015     # 1.5% filas duplicadas in-place

print(f'Configuración cargada. {N_REGISTROS:,} registros por dataset.')

Configuración cargada. 450,000 registros por dataset.


## 3. Cargar muestras del Notebook 01

In [3]:
# Cargar las 4 muestras generadas
muestras = {}
for anio in [2021, 2022, 2023, 2024]:
    ruta = os.path.join(CARPETA, f'muestra_{anio}.csv')
    if not os.path.exists(ruta):
        raise FileNotFoundError(f'No se encontró {ruta}. Ejecuta primero el Notebook 01.')
    muestras[anio] = pd.read_csv(ruta, sep=',', encoding='utf-8', dtype=str, low_memory=False)
    print(f'muestra_{anio}.csv: {len(muestras[anio]):,} registros, {len(muestras[anio].columns)} columnas')

gc.collect()

muestra_2021.csv: 450,000 registros, 27 columnas
muestra_2022.csv: 450,000 registros, 27 columnas
muestra_2023.csv: 450,000 registros, 27 columnas
muestra_2024.csv: 450,000 registros, 27 columnas


23

## 4. Crear datasets base (2020 y 2025)

- **dataset_2020:** 225.000 registros de 2021 + 225.000 de 2022
- **dataset_2025:** 225.000 registros de 2023 + 225.000 de 2024

Se ajusta `FECHA_ATENCION` al año correspondiente y se recalcula `EDAD_ANIOS`.

In [4]:
def crear_dataset_base(muestra_a, anio_a, muestra_b, anio_b, anio_destino):
    """Combina dos muestras y ajusta fechas al año destino."""
    # Tomar N_POR_ANIO registros aleatorios de cada año
    parte_a = muestra_a.sample(n=N_POR_ANIO, random_state=RANDOM_STATE).reset_index(drop=True)
    parte_b = muestra_b.sample(n=N_POR_ANIO, random_state=RANDOM_STATE + 1).reset_index(drop=True)

    # Concatenar
    df = pd.concat([parte_a, parte_b], ignore_index=True)
    print(f'  Registros combinados: {len(df):,} ({N_POR_ANIO:,} de {anio_a} + {N_POR_ANIO:,} de {anio_b})')

    # Ajustar FECHA_ATENCION al año destino
    # Extraer mes y día de la fecha original, cambiar el año
    def ajustar_fecha(fecha_str, anio_nuevo):
        if not isinstance(fecha_str, str) or pd.isna(fecha_str):
            return fecha_str
        partes = fecha_str.split('-')
        if len(partes) == 3:
            return f'{anio_nuevo}-{partes[1]}-{partes[2]}'
        partes = fecha_str.split('/')
        if len(partes) == 3:
            if len(partes[2]) == 4:  # dd/mm/yyyy
                return f'{anio_nuevo}-{partes[1]}-{partes[0]}'
            elif len(partes[0]) == 4:  # yyyy/mm/dd
                return f'{anio_nuevo}-{partes[1]}-{partes[2]}'
        return fecha_str

    df['FECHA_ATENCION'] = df['FECHA_ATENCION'].apply(lambda x: ajustar_fecha(x, anio_destino))

    # Recalcular EDAD_ANIOS basado en la diferencia entre año destino y año de nacimiento
    def recalcular_edad(fecha_nac_str, anio_atencion):
        if not isinstance(fecha_nac_str, str) or pd.isna(fecha_nac_str):
            return None
        try:
            partes = fecha_nac_str.split('/')
            if len(partes) == 3:
                anio_nac = int(partes[2])
            else:
                partes = fecha_nac_str.split('-')
                anio_nac = int(partes[0])
            edad = anio_atencion - anio_nac
            return str(edad) if edad >= 0 else None
        except (ValueError, IndexError):
            return None

    df['EDAD_ANIOS'] = df['FECHA_NACIMIENTO'].apply(lambda x: recalcular_edad(x, anio_destino))

    return df


# Crear dataset 2020 (desde 2021 + 2022)
print('Creando dataset 2020...')
df_2020 = crear_dataset_base(muestras[2021], 2021, muestras[2022], 2022, 2020)

# Crear dataset 2025 (desde 2023 + 2024)
print('\nCreando dataset 2025...')
df_2025 = crear_dataset_base(muestras[2023], 2023, muestras[2024], 2024, 2025)

# Liberar las muestras originales
del muestras
gc.collect()

print(f'\nDatasets base creados:')
print(f'  2020: {len(df_2020):,} registros')
print(f'  2025: {len(df_2025):,} registros')

Creando dataset 2020...
  Registros combinados: 450,000 (225,000 de 2021 + 225,000 de 2022)

Creando dataset 2025...
  Registros combinados: 450,000 (225,000 de 2023 + 225,000 de 2024)

Datasets base creados:
  2020: 450,000 registros
  2025: 450,000 registros


## 5. Funciones para aplicar inconsistencias

Las inconsistencias son realistas y de baja complejidad, suitable para practica de limpieza en KNIME.

In [5]:
def aplicar_inconsistencias(df, seed=42):
    """Aplica inconsistencias realistas al dataset.
    Modifica el dataframe in-place y retorna una descripción de los cambios.
    """
    rng = np.random.RandomState(seed)
    n = len(df)
    descripcion = []

    # ========================================================================
    # 1. VALORES NULOS (~2.5%)
    # Se reemplazan celdas aleatorias por NaN en columnas de texto
    # ========================================================================
    cols_nulos = ['NACIONALIDAD', 'AUTOIDENTIFICACION_ETNICA',
                  'GRUPO_PRIORITARIO_1', 'GRUPO_PRIORITARIO_2', 'GRUPO_PRIORITARIO_3',
                  'CODIGO_CIE10_2', 'DESCRIPCION_CIE10_2',
                  'CODIGO_CIE10_3', 'DESCRIPCION_CIE10_3',
                  'CANTON_RESIDENCIA']
    total_nulos = 0
    for col in cols_nulos:
        if col in df.columns:
            # Seleccionar ~25% de las filas para esta columna (distribuir entre columnas)
            mascara = rng.random(n) < 0.15  # ~15% de filas por columna
            n_afectadas = mascara.sum()
            df.loc[mascara, col] = np.nan
            total_nulos += n_afectadas
    descripcion.append(f'Valores nulos: celdas vaciadas en columnas de texto secundarias (~{total_nulos:,} celdas)')
    print(f'  1. Valores nulos aplicados: ~{total_nulos:,} celdas en {len(cols_nulos)} columnas')

    # ========================================================================
    # 2. ESPACIOS EXTRA AL INICIO/FINAL (~2%)
    # Se agregan espacios en blanco antes o después del texto
    # ========================================================================
    cols_espacios = ['ZONA', 'PROVINCIA', 'CANTON', 'SEXO', 'NACIONALIDAD',
                     'AUTOIDENTIFICACION_ETNICA', 'PROVINCIA_RESIDENCIA', 'CANTON_RESIDENCIA']
    total_espacios = 0
    for col in cols_espacios:
        if col in df.columns:
            mascara = (rng.random(n) < 0.08) & df[col].notna()  # ~8% por columna
            indices = df.index[mascara]
            for idx in indices:
                val = df.at[idx, col]
                if isinstance(val, str) and val != '':
                    tipo = rng.choice(['izq', 'der', 'ambos'])
                    if tipo == 'izq':
                        df.at[idx, col] = '  ' + val
                    elif tipo == 'der':
                        df.at[idx, col] = val + '  '
                    else:
                        df.at[idx, col] = ' ' + val + ' '
                    total_espacios += 1
    descripcion.append(f'Espacios extra: ~{total_espacios:,} valores con espacios al inicio/final')
    print(f'  2. Espacios extra aplicados: ~{total_espacios:,} valores')

    # ========================================================================
    # 3. MAYÚSCULAS / MINÚSCULAS MIXTAS (~2%)
    # Capitalizacion incorrecta: Pichincha en vez de PICHINCHA, guayas en vez de GUAYAS
    # ========================================================================
    cols_mayus = ['PROVINCIA', 'CANTON', 'PROVINCIA_RESIDENCIA', 'CANTON_RESIDENCIA',
                  'DESCRIPCION_CIE10_1', 'DESCRIPCION_CIE10_2', 'DESCRIPCION_CIE10_3']
    total_mayus = 0
    for col in cols_mayus:
        if col in df.columns:
            mascara = (rng.random(n) < 0.08) & df[col].notna()
            indices = df.index[mascara]
            for idx in indices:
                val = df.at[idx, col]
                if isinstance(val, str) and val != '':
                    tipo = rng.choice(['title', 'lower', 'capitalize'])
                    if tipo == 'title':
                        df.at[idx, col] = val.title()      # "PICHINCHA" -> "Pichincha"
                    elif tipo == 'lower':
                        df.at[idx, col] = val.lower()      # "GUAYAS" -> "guayas"
                    else:
                        df.at[idx, col] = val.capitalize() # "GUAYAS" -> "Guayas"
                    total_mayus += 1
    descripcion.append(f'Mayúsculas/minúsculas: ~{total_mayus:,} valores con capitalización inconsistente')
    print(f'  3. Capitalización inconsistente aplicada: ~{total_mayus:,} valores')

    # ========================================================================
    # 4. VALORES DE REEMPLAZO: "No registra", "N/A", "-", "" (~2%)
    # Se reemplazan valores válidos por estos tokens genéricos
    # ========================================================================
    tokens_reemplazo = ['No registra', 'N/A', '-', '']
    cols_reemplazo = ['AUTOIDENTIFICACION_ETNICA', 'GRUPO_PRIORITARIO_1',
                      'GRUPO_PRIORITARIO_2', 'GRUPO_PRIORITARIO_3',
                      'NACIONALIDAD', 'CODIGO_CIE10_2', 'CODIGO_CIE10_3']
    total_reemplazos = 0
    for col in cols_reemplazo:
        if col in df.columns:
            mascara = (rng.random(n) < 0.08) & df[col].notna()
            n_reemplazar = mascara.sum()
            tokens = rng.choice(tokens_reemplazo, size=n_reemplazar)
            df.loc[mascara, col] = tokens
            total_reemplazos += n_reemplazar
    descripcion.append(f'Valores reemplazados: ~{total_reemplazos:,} celdas por "No registra", "N/A", "-", ""')
    print(f'  4. Valores de reemplazo aplicados: ~{total_reemplazos:,} celdas')

    # ========================================================================
    # RESUMEN DE INCONSISTENCIAS
    # ========================================================================
    total_celdas = n * len(df.columns)
    total_inconsistencias = total_nulos + total_espacios + total_mayus + total_reemplazos
    pct_real = (total_inconsistencias / total_celdas) * 100
    print(f'\n  Total de celdas modificadas: ~{total_inconsistencias:,} de {total_celdas:,} ({pct_real:.2f}%)')

    return descripcion

print('Función de inconsistencias definida.')

Función de inconsistencias definida.


## 6. Función para duplicar filas in-place

Se seleccionan ~1.5% de filas al azar, se duplican y se insertan inmediatamente después de la original.
Para mantener el conteo exacto de 450.000, se eliminan filas del final.

In [6]:
def duplicar_filas_inplace(df, pct=0.015, seed=42):
    """Duplica un porcentaje de filas y las inserta in-place.
    Mantiene el total de registros en 450.000 eliminando filas del final.
    """
    rng = np.random.RandomState(seed)
    n = len(df)
    n_duplicar = int(n * pct)

    # Seleccionar índices aleatorios para duplicar
    indices_duplicar = sorted(rng.choice(n, size=n_duplicar, replace=False))

    # Crear las filas duplicadas
    duplicados = df.iloc[indices_duplicar].copy()

    # Insertar cada duplicado justo después de su original
    # Para eficiencia, reconstruimos el dataframe
    filas_finales = []
    conjunto_duplicar = set(indices_duplicar)
    for i in range(n):
        filas_finales.append(df.iloc[i])
        if i in conjunto_duplicar:
            filas_finales.append(duplicados.loc[i])

    df_nuevo = pd.DataFrame(filas_finales).reset_index(drop=True)

    # Mantener exactamente 450.000 registros (recortar excedente)
    if len(df_nuevo) > N_REGISTROS:
        df_nuevo = df_nuevo.iloc[:N_REGISTROS].reset_index(drop=True)
    elif len(df_nuevo) < N_REGISTROS:
        # Si quedan menos, duplicar más filas para compensar
        deficit = N_REGISTROS - len(df_nuevo)
        extras = rng.choice(n, size=deficit, replace=True)
        df_extras = df.iloc[extras].reset_index(drop=True)
        df_nuevo = pd.concat([df_nuevo, df_extras], ignore_index=True)

    print(f'  Duplicadas {n_duplicar:,} filas ({pct*100:.1f}%). Total final: {len(df_nuevo):,}')
    return df_nuevo

print('Función de duplicados definida.')

Función de duplicados definida.


## 7. Aplicar inconsistencias y duplicados al dataset 2020

In [7]:
print('='*60)
print('PROCESANDO DATASET 2020')
print('='*60)

# Aplicar inconsistencias
print('\nAplicando inconsistencias...')
desc_2020 = aplicar_inconsistencias(df_2020, seed=42)

# Duplicar filas in-place
print('\nDuplicando filas...')
df_2020 = duplicar_filas_inplace(df_2020, pct=PCT_DUPLICADOS, seed=42)

print(f'\nShape final: {df_2020.shape}')

PROCESANDO DATASET 2020

Aplicando inconsistencias...
  1. Valores nulos aplicados: ~674,611 celdas en 10 columnas
  2. Espacios extra aplicados: ~269,260 valores
  3. Capitalización inconsistente aplicada: ~212,920 valores
  4. Valores de reemplazo aplicados: ~213,680 celdas

  Total de celdas modificadas: ~1,370,471 de 12,150,000 (11.28%)

Duplicando filas...
  Duplicadas 6,750 filas (1.5%). Total final: 450,000

Shape final: (450000, 27)


## 8. Aplicar inconsistencias y duplicados al dataset 2025

In [8]:
print('='*60)
print('PROCESANDO DATASET 2025')
print('='*60)

# Aplicar inconsistencias
print('\nAplicando inconsistencias...')
desc_2025 = aplicar_inconsistencias(df_2025, seed=99)

# Duplicar filas in-place
print('\nDuplicando filas...')
df_2025 = duplicar_filas_inplace(df_2025, pct=PCT_DUPLICADOS, seed=99)

print(f'\nShape final: {df_2025.shape}')

PROCESANDO DATASET 2025

Aplicando inconsistencias...
  1. Valores nulos aplicados: ~675,047 celdas en 10 columnas
  2. Espacios extra aplicados: ~269,385 valores
  3. Capitalización inconsistente aplicada: ~233,518 valores
  4. Valores de reemplazo aplicados: ~214,302 celdas

  Total de celdas modificadas: ~1,392,252 de 12,150,000 (11.46%)

Duplicando filas...
  Duplicadas 6,750 filas (1.5%). Total final: 450,000

Shape final: (450000, 27)


## 9. Guardar datasets sucios

In [9]:
for anio, df in [(2020, df_2020), (2025, df_2025)]:
    nombre = f'dataset_{anio}_sucio.csv'
    ruta = os.path.join(CARPETA, nombre)
    df.to_csv(ruta, index=False, encoding='utf-8')
    print(f'Guardado: {nombre} ({len(df):,} registros, {len(df.columns)} columnas)')

# Liberar memoria
del df_2020, df_2025
gc.collect()

print('\nProceso completado.')

Guardado: dataset_2020_sucio.csv (450,000 registros, 27 columnas)
Guardado: dataset_2025_sucio.csv (450,000 registros, 27 columnas)

Proceso completado.


## 10. Resumen de inconsistencias aplicadas

In [10]:
print('RESUMEN DE INCONSISTENCIAS')
print('='*60)
print('\nDataset 2020:')
for desc in desc_2020:
    print(f'  - {desc}')

print('\nDataset 2025:')
for desc in desc_2025:
    print(f'  - {desc}')

print('\n' + '='*60)
print('Archivos generados:')
print('  - dataset_2020_sucio.csv')
print('  - dataset_2025_sucio.csv')
print('\nListos para limpieza en KNIME.')

RESUMEN DE INCONSISTENCIAS

Dataset 2020:
  - Valores nulos: celdas vaciadas en columnas de texto secundarias (~674,611 celdas)
  - Espacios extra: ~269,260 valores con espacios al inicio/final
  - Mayúsculas/minúsculas: ~212,920 valores con capitalización inconsistente
  - Valores reemplazados: ~213,680 celdas por "No registra", "N/A", "-", ""

Dataset 2025:
  - Valores nulos: celdas vaciadas en columnas de texto secundarias (~675,047 celdas)
  - Espacios extra: ~269,385 valores con espacios al inicio/final
  - Mayúsculas/minúsculas: ~233,518 valores con capitalización inconsistente
  - Valores reemplazados: ~214,302 celdas por "No registra", "N/A", "-", ""

Archivos generados:
  - dataset_2020_sucio.csv
  - dataset_2025_sucio.csv

Listos para limpieza en KNIME.
